# Module 3: Inference Optimization + Model Export

A model that trains well is only half the story. In production, **inference** is what users interact with.

In this module, you'll:

1. **Benchmark** baseline inference latency
2. **Quantize** the model (dynamic quantization) for smaller size and faster CPU inference
3. **Export** to TorchScript and ONNX for deployment outside Python
4. **Compare** all approaches on latency and model size

---

In [ ]:
# --- Colab / Environment Setup (run this cell first) ---
import os, subprocess

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    if not os.path.exists("/content/pytorch-production-workshop"):
        subprocess.run(["git", "clone", "https://github.com/arj7192/pytorch-production-workshop.git"], cwd="/content", check=True)
    os.chdir("/content/pytorch-production-workshop/notebooks")
    subprocess.run(["pip", "install", "-q", "-r", "../requirements.txt"], check=True)
    print("Colab setup complete - GPU:", os.environ.get("COLAB_GPU", "not detected"))

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import torch
import torch.nn as nn
from pathlib import Path

from src.model import build_model
from src.data import prepare_wikitext2
from src.evaluate import benchmark_inference, generate_sample
from src.export import export_torchscript, export_onnx, verify_torchscript, verify_onnx
from src.utils import set_seed, get_device, CheckpointManager

set_seed(42)
device = get_device()
print(f"Device: {device}")

In [ ]:
# Load trained model from Module 2's checkpoint.
# If you're starting fresh (no checkpoint from Module 2), a quick fallback
# training run kicks in automatically so this notebook is self-contained.
train_dataset, val_dataset, _, tokenizer = prepare_wikitext2(
    vocab_size=8192, seq_len=128, tokenizer_path='../tokenizer.json'
)

config = {
    'vocab_size': tokenizer.get_vocab_size(),
    'd_model': 256, 'n_heads': 4, 'd_ff': 512,
    'n_layers': 4, 'max_seq_len': 128, 'dropout': 0.1,
}

model = build_model(config).to(device)

# Load checkpoint from Module 2; if not available, do a quick training run
ckpt_manager = CheckpointManager('../checkpoints')
latest = ckpt_manager.latest()
if latest:
    ckpt_manager.load(model, path=latest)
    print(f"Loaded checkpoint: {latest}")
else:
    print("No checkpoint found - running a quick training pass...")
    from src.data import create_dataloaders
    from src.evaluate import evaluate
    loader, val_loader = create_dataloaders(train_dataset, val_dataset, batch_size=64)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda') if use_amp else None
    for epoch in range(3):
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp, dtype=torch.float16):
                loss = model(x, targets=y)['loss']
            if scaler:
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            opt.zero_grad(set_to_none=True)
        val_m = evaluate(model, val_loader, device, use_amp=use_amp)
        print(f"  Epoch {epoch+1}/3 | val_loss {val_m['val_loss']:.4f} | val_ppl {val_m['val_perplexity']:.1f}")
    ckpt_manager.save(model, opt, 3, 0, val_m['val_loss'], config)
    latest = ckpt_manager.latest()
    print(f"  Checkpoint saved: {latest}")

model.eval()
print(f"Model: {model.count_parameters():,} parameters")

## 3.1 Baseline Inference Benchmark

**Rule #1 of optimization: measure before you optimize.**

We benchmark three things:
- **Single sample latency** - What one user request costs
- **Batched inference throughput** - What happens when you process many requests at once
- **Percentiles (p50, p95, p99)** - Production SLAs are written as p99 ("99% of requests under 50ms"), not mean. A mean of 5ms is useless if 1% of requests take 500ms.

**Common benchmarking mistakes we avoid here:**
1. Missing `torch.cuda.synchronize()` - CUDA ops are async. Without sync, you measure kernel launch time, not compute time. Your numbers look artificially fast.
2. No warmup - The first few runs include JIT compilation, CUDA context init, and memory pool setup. Not representative of steady state.
3. Reporting only the mean - Hides tail latency from GC pauses, memory allocation, and OS scheduling.

In [ ]:
# Create test inputs (random token IDs - content doesn't matter for latency benchmarks)
seq_len = 128
single_input = torch.randint(0, config['vocab_size'], (1, seq_len)).to(device)
batch_input = torch.randint(0, config['vocab_size'], (32, seq_len)).to(device)

# benchmark_inference does 10 warmup runs, then 200 measured runs with
# torch.cuda.synchronize() around each timing. Reports mean, p50, p95, p99.
single_stats = benchmark_inference(model, single_input, device, n_runs=200)
print("=== Single Sample Inference ===")
print(f"  Mean: {single_stats['mean_ms']:.2f} ms")
print(f"  P50:  {single_stats['p50_ms']:.2f} ms")
print(f"  P95:  {single_stats['p95_ms']:.2f} ms")
print(f"  P99:  {single_stats['p99_ms']:.2f} ms")

# Batched inference: the biggest throughput trick.
# A GPU has thousands of cores. Processing 1 sample uses a fraction of them.
# Processing 32 is a wider matrix multiply that fills more cores.
# Watch the per-sample cost drop dramatically.
batch_stats = benchmark_inference(model, batch_input, device, n_runs=100)
print(f"\n=== Batched Inference (batch_size=32) ===")
print(f"  Mean: {batch_stats['mean_ms']:.2f} ms")
print(f"  Per sample: {batch_stats['mean_ms'] / 32:.2f} ms")
print(f"  Throughput: {32 / batch_stats['mean_ms'] * 1000:.0f} samples/sec")

## 3.2 Dynamic Quantization

One function call. ~4x smaller model. 1.5-3x faster CPU inference. Barely any accuracy loss.

Quantization converts weight matrices from float32 (32 bits, 4 bytes) to int8 (8 bits, 1 byte). A float32 weight of 0.0347 becomes integer 9 with a scale factor that maps it back to approximately 0.035. You lose some precision, but for millions of weights, the tiny rounding errors largely cancel out.

**Why it's faster on CPU**: (1) Reading 5 MB from RAM is 4x faster than reading 20 MB - and memory bandwidth is usually the CPU bottleneck, not arithmetic. (2) Modern CPUs have native int8 instructions (VNNI on Intel, NEON on ARM) that are faster than float32 math.

**"Dynamic" means**: Weights are quantized ahead of time (when you call `quantize_dynamic`), but activations are quantized on the fly during each forward pass. The alternative, "static" quantization, pre-calibrates activation ranges using a calibration dataset - slightly faster but more setup.

**We only quantize `nn.Linear` layers** because that's where 95% of the parameters and compute are in a transformer. Embeddings are lookup tables (not matmuls), and LayerNorm has very few parameters.

> **GPU quantization** is a different world: INT8 via TensorRT, INT4 via bitsandbytes/GPTQ. Those are about fitting larger models into limited VRAM, not about speed on CPU.

In [ ]:
# Move to CPU for quantization
model_cpu = build_model(config)
if latest:
    ckpt_manager.load(model_cpu, path=latest)
model_cpu.eval()

# One function call: float32 weights -> int8 weights for all nn.Linear layers.
# Each attention layer has 4 Linear ops (Q, K, V, output) + 2 FFN (up, down).
# That's 6 per layer x 4 layers = 24 Linear layers quantized.
model_quantized = torch.ao.quantization.quantize_dynamic(
    model_cpu,
    {nn.Linear},
    dtype=torch.qint8,
)

# Compare sizes
import tempfile, os

def model_size_mb(model):
    with tempfile.NamedTemporaryFile(delete=False) as f:
        torch.save(model.state_dict(), f.name)
        size = os.path.getsize(f.name) / 1024 / 1024
        os.unlink(f.name)
    return size

orig_size = model_size_mb(model_cpu)
quant_size = model_size_mb(model_quantized)

print(f"Original model:   {orig_size:.1f} MB")
print(f"Quantized model:  {quant_size:.1f} MB")
print(f"Size reduction:   {(1 - quant_size/orig_size) * 100:.0f}%")

In [ ]:
# Benchmark quantized vs original on CPU
cpu_device = torch.device('cpu')
test_input_cpu = torch.randint(0, config['vocab_size'], (1, seq_len))

orig_stats = benchmark_inference(model_cpu, test_input_cpu, cpu_device, n_runs=200)
quant_stats = benchmark_inference(model_quantized, test_input_cpu, cpu_device, n_runs=200)

print(f"{'Metric':<15} {'Original':>12} {'Quantized':>12} {'Speedup':>10}")
print('-' * 52)
for metric in ['mean_ms', 'p50_ms', 'p95_ms']:
    orig_val = orig_stats[metric]
    quant_val = quant_stats[metric]
    speedup = orig_val / quant_val
    label = metric.replace('_ms', '')
    print(f"{label:<15} {orig_val:>10.2f}ms {quant_val:>10.2f}ms {speedup:>9.2f}x")

## 3.3 TorchScript Export

TorchScript is PyTorch's answer to "how do I run this model without Python?" The exported `.pt` file contains the computation graph AND the weights. It can be:
- **Loaded in C++** via `libtorch` - no Python interpreter, no GIL, no import overhead
- **Shipped as a single file** to mobile, embedded, or any C++ service
- **Optimized** by the JIT compiler (constant folding, dead code elimination)

**Two approaches:**
- **`torch.jit.trace`**: Runs the model once with a sample input and records every tensor operation. Fast, simple, but can't capture input-dependent `if/else` branches - it only records whichever path executed during tracing. For inference with fixed architecture and no branching, this is almost always the right choice.
- **`torch.jit.script`**: Analyzes the Python source code and compiles it to TorchScript IR. Handles control flow but has strict type requirements and many Python features aren't supported.

**Why do we need an `InferenceWrapper`?** Our model returns a dict `{'logits': ..., 'loss': ...}`. `torch.jit.trace` can't handle dict returns (it only records tensor operations). The wrapper extracts just the logits. This is a common production pattern: training models have rich interfaces (loss, metrics, attention weights), inference models are minimal (tensor in, tensor out).

In [ ]:
# Wrapper strips the dict return to a single tensor (logits only).
# Training: rich interface. Inference: minimal interface.
class InferenceWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        return self.model(input_ids)['logits']

wrapper = InferenceWrapper(model_cpu)
wrapper.eval()

sample_input = torch.randint(0, config['vocab_size'], (1, seq_len))

# Trace: run once, record all tensor operations
traced_path = export_torchscript(
    wrapper, sample_input,
    output_path='../exports/model_traced.pt',
    method='trace',
)

# Verify: same input through original and exported model, check outputs match
# within floating-point tolerance (1e-5). A max diff > 1e-4 means something
# went wrong during the export.
verify_torchscript(wrapper, traced_path, sample_input)

In [ ]:
# Load and benchmark TorchScript model
ts_model = torch.jit.load(traced_path)
ts_model.eval()

ts_stats = benchmark_inference(ts_model, test_input_cpu, cpu_device, n_runs=200)

print(f"TorchScript mean latency: {ts_stats['mean_ms']:.2f} ms")
print(f"Original mean latency:    {orig_stats['mean_ms']:.2f} ms")
print(f"Speedup: {orig_stats['mean_ms'] / ts_stats['mean_ms']:.2f}x")

## 3.4 ONNX Export

**ONNX is the USB-C of ML models** - one format that runs everywhere.

ONNX defines ~170 standard operators (MatMul, Softmax, LayerNorm, etc.) and a binary protobuf format. Any framework can export to it, any runtime can execute it:
- **ONNX Runtime** (Microsoft) - best general-purpose CPU/GPU performance
- **TensorRT** (NVIDIA) - maximum GPU performance, NVIDIA-only
- **OpenVINO** (Intel) - optimized for Intel CPUs, VPUs, FPGAs
- **CoreML** (Apple) - deploy to iOS, macOS, Apple Neural Engine

**Why is ONNX Runtime faster than PyTorch on CPU?** Three reasons: (1) Graph optimization - fuse MatMul+Add+ReLU into one kernel. (2) Optimized kernels from Intel MKL-DNN. (3) Pure C++ inference loop - no Python interpreter, no GIL, no garbage collection pauses.

**Key detail**: the `dynamic_axes` argument in our export tells ONNX that batch size and sequence length can vary at inference time. Without it, the model only accepts the exact shape used during export.

In [ ]:
# Export to ONNX
onnx_path = export_onnx(
    wrapper, sample_input,
    output_path='../exports/model.onnx',
)

# Verify
try:
    verify_onnx(wrapper, onnx_path, sample_input)
except ImportError:
    print("onnxruntime not installed - install with: pip install onnxruntime")

In [ ]:
# Benchmark ONNX Runtime inference
try:
    import onnxruntime as ort
    import numpy as np
    
    session = ort.InferenceSession(onnx_path)
    
    # Warmup
    np_input = test_input_cpu.numpy()
    for _ in range(10):
        session.run(None, {'input_ids': np_input})
    
    # Benchmark
    latencies = []
    for _ in range(200):
        start = time.perf_counter()
        session.run(None, {'input_ids': np_input})
        latencies.append((time.perf_counter() - start) * 1000)
    
    latencies.sort()
    print(f"ONNX Runtime mean latency: {sum(latencies)/len(latencies):.2f} ms")
    print(f"ONNX Runtime P50:          {latencies[len(latencies)//2]:.2f} ms")
    print(f"PyTorch mean latency:      {orig_stats['mean_ms']:.2f} ms")
    print(f"Speedup: {orig_stats['mean_ms'] / (sum(latencies)/len(latencies)):.2f}x")
    
except ImportError:
    print("onnxruntime not installed - skipping benchmark")

## 3.5 Comparison Summary

There is no universally "best" format. The right choice depends on your deployment target:

- **GPU serving in the cloud?** PyTorch eager + `torch.compile`. Fast enough, maximum flexibility.
- **CPU serving for cost savings?** ONNX Runtime + quantization. Best CPU performance.
- **Mobile or embedded?** TorchScript + quantization. No Python dependency, small file size.
- **Multi-platform?** Export to ONNX - it's the bridge between frameworks and hardware.
- **Not sure yet?** Just serve PyTorch eager behind FastAPI. Optimize when you have production data.

In [ ]:
# Collect all results
results = {
    'PyTorch (FP32)': {
        'latency_ms': orig_stats['mean_ms'],
        'size_mb': orig_size,
    },
    'Quantized (INT8)': {
        'latency_ms': quant_stats['mean_ms'],
        'size_mb': quant_size,
    },
    'TorchScript': {
        'latency_ms': ts_stats['mean_ms'],
        'size_mb': Path(traced_path).stat().st_size / 1024 / 1024,
    },
}

try:
    results['ONNX Runtime'] = {
        'latency_ms': sum(latencies) / len(latencies),
        'size_mb': Path(onnx_path).stat().st_size / 1024 / 1024,
    }
except NameError:
    pass

baseline_latency = results['PyTorch (FP32)']['latency_ms']

print(f"{'Format':<20} {'Latency (ms)':>14} {'Size (MB)':>10} {'Speedup':>10}")
print('=' * 58)
for name, data in results.items():
    speedup = baseline_latency / data['latency_ms']
    print(f"{name:<20} {data['latency_ms']:>12.2f}ms {data['size_mb']:>8.1f}MB {speedup:>9.2f}x")

## Key Takeaways

| Export format | Best for | Trade-off |
|--------------|---------|----------|
| **PyTorch eager** | Prototyping, GPU serving | Slowest on CPU, most flexible |
| **Dynamic quantization** | CPU deployment | 2-4x smaller, 1.5-3x faster on CPU |
| **TorchScript** | C++ deployment, mobile | No Python needed, single-file deployment |
| **ONNX** | Multi-platform inference | Best CPU perf via ORT, widest hardware support |

**Production recommendation**: Use ONNX Runtime for CPU serving, quantized models for edge/mobile, and keep PyTorch eager for GPU serving (it's already fast on CUDA).

**Remember:**
- Always benchmark with warmup and report percentiles, not just mean
- Batching is often a bigger win than any export format
- Verify every export - run the same input through both models and check the outputs match
- Quantization + ONNX can be combined for maximum CPU performance

**Next up**: Module 4 - deploying the model as a FastAPI service with Docker and Cloud Run.